# 02 Junyi prerequisite graph (C1 Adaptive Curriculum Engine)

Builds the directed prerequisite graph the GAT is trained on, and checks that it is trustworthy. This is the evidence for proposal **Appendix B**, done properly:

1. **Edges counted correctly.** A `prerequisites` cell can hold several exercises separated by commas, so edges are counted after splitting, not by counting cells.
2. **Graph checks.** Edges pointing at unknown exercises, duplicates, self-loops and cycles.
3. **Direction check.** Do the annotations rate "A is a prerequisite of B" higher for real edges A -> B than for the reverse and for unrelated pairs? This tells us which way the annotation pairs read and whether the edges agree with the experts.
4. **Annotation reliability.** Krippendorff's alpha over the raw expert ratings (it handles a different number of raters per pair and does not need to know which rater is which).
5. **Export** of the edge list in the exercise numbering from notebook 01, ready for GAT training.

Runs on CPU in a few minutes. Needs notebook 01 to have run (it reuses `exercise_index.json`).

In [1]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"

In [2]:
import sys
!{sys.executable} -m pip install -q EduData pyarrow networkx scipy

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

RAW = BASE / "raw" / "junyi"
PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "graph"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not (RAW / "junyi_Exercise_table.csv").exists():
    from EduData import get_data
    get_data("junyi", str(BASE / "raw"))
print("Storage:", STORAGE, "->", BASE)

Mounted at /content/drive
Storage: drive -> /content/drive/MyDrive/adaptlearn-c1


## 1. Exercises and edges, counted correctly

In [3]:
import json
import numpy as np
import pandas as pd

SEED = 42
ex = pd.read_csv(RAW / "junyi_Exercise_table.csv", low_memory=False)
live = ex[ex["live"] == True].copy()
names = set(live["name"])
print(f"exercises: {len(ex)}  live: {len(live)}  with a prerequisites value: {live['prerequisites'].notna().sum()}")

raw_edges = (
    live.dropna(subset=["prerequisites"])
        .assign(prerequisite=lambda d: d["prerequisites"].str.split(","))
        .explode("prerequisite")
        .assign(prerequisite=lambda d: d["prerequisite"].str.strip())
        [["prerequisite", "name", "area"]]
        .rename(columns={"name": "exercise"})
)
raw_edges = raw_edges[raw_edges["prerequisite"] != ""]
print("edges after splitting multi-prerequisite cells:", len(raw_edges))

exercises: 837  live: 776  with a prerequisites value: 715
edges after splitting multi-prerequisite cells: 950


In [4]:
checks = []
dangling = raw_edges[~raw_edges["prerequisite"].isin(names)]
checks.append(("point at an exercise that is not live or not in the table", len(dangling)))
edges = raw_edges[raw_edges["prerequisite"].isin(names)]

self_loops = edges[edges["prerequisite"] == edges["exercise"]]
checks.append(("self-loops", len(self_loops)))
edges = edges[edges["prerequisite"] != edges["exercise"]]

dupes = edges.duplicated(subset=["prerequisite", "exercise"]).sum()
checks.append(("duplicates", int(dupes)))
edges = edges.drop_duplicates(subset=["prerequisite", "exercise"]).reset_index(drop=True)

display(pd.DataFrame(checks, columns=["removed edges that", "count"]))
print("clean edges:", len(edges))

,removed edges that,count
0,point at an exercise that is not live or not i...,14
1,self-loops,2
2,duplicates,1


clean edges: 933


## 2. Graph structure

In [5]:
import networkx as nx

G = nx.DiGraph()
G.add_nodes_from(sorted(names))
G.add_edges_from(zip(edges["prerequisite"], edges["exercise"]))

is_dag = nx.is_directed_acyclic_graph(G)
print("no cycles:", is_dag)
if not is_dag:
    cycles = list(nx.simple_cycles(G))
    print(f"{len(cycles)} cycles, e.g.", cycles[:3])

area = live.set_index("name")["area"].to_dict()
cross_area = sum(area.get(p) != area.get(c) for p, c in G.edges())
roots = [n for n in G if G.in_degree(n) == 0]
isolated = [n for n in G if G.degree(n) == 0]
longest = nx.dag_longest_path_length(G) if is_dag else None
# An edge is redundant when a longer path already implies it (A->B->C makes A->C redundant).
redundant = sorted(set(G.edges()) - set(nx.transitive_reduction(G).edges())) if is_dag else []

graph_stats = pd.Series({
    "nodes (live exercises)": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "edges across areas": cross_area,
    "starting exercises (no prerequisite)": len(roots),
    "isolated exercises": len(isolated),
    "longest prerequisite chain": longest,
    "redundant edges (implied by a longer path)": len(redundant),
    "mean prerequisites per exercise": round(np.mean([d for _, d in G.in_degree()]), 2),
})
graph_stats

no cycles: False
1 cycles, e.g. [['adding_and_subtracting_radicals', 'radical_multiplication_and_division', 'simplifying_radicals']]


,0
nodes (live exercises),776.0
edges,933.0
edges across areas,43.0
starting exercises (no prerequisite),69.0
isolated exercises,29.0
longest prerequisite chain,NaN
redundant edges (implied by a longer path),0.0
mean prerequisites per exercise,1.2


## 3. Direction and agreement with the expert annotations

`Prerequisite_avg` is the mean rating (1 to 9) the annotators gave to "one exercise of the pair is a prerequisite of the other". Each annotated pair is classified as:

- **edge A->B**: `Exercise_A -> Exercise_B` is an edge in the graph
- **edge B->A**: the reverse is an edge
- **not an edge**: neither direction

If edges agree with the experts, both edge groups score clearly higher than "not an edge". The group that scores higher between the two edge groups tells us which way the annotation question reads.

In [6]:
from scipy.stats import mannwhitneyu

rel = pd.concat([
    pd.read_csv(RAW / "relationship_annotation_training.csv"),
    pd.read_csv(RAW / "relationship_annotation_testing.csv"),
], ignore_index=True)
E = set(G.edges())
rel["kind"] = np.select(
    [[(a, b) in E for a, b in zip(rel.Exercise_A, rel.Exercise_B)],
     [(b, a) in E for a, b in zip(rel.Exercise_A, rel.Exercise_B)]],
    ["edge A->B", "edge B->A"], "not an edge")

summary = rel.groupby("kind")["Prerequisite_avg"].agg(["count", "mean", "median", "std"]).round(2)
display(summary)

is_edge = rel["kind"] != "not an edge"
u, p = mannwhitneyu(rel.loc[is_edge, "Prerequisite_avg"], rel.loc[~is_edge, "Prerequisite_avg"], alternative="greater")
print(f"Edges rated higher than non-edges: Mann-Whitney U = {u:.0f}, one-sided p = {p:.2e}")
strong = (rel.loc[is_edge, "Prerequisite_avg"] >= 6).mean()
print(f"Share of annotated edges rated strong (>= 6): {strong:.1%}")

,count,mean,median,std
kind,,,,
edge A->B,256,5.16,5.05,1.63
edge B->A,374,6.14,6.55,1.92
not an edge,1324,4.04,3.67,2.05


Edges rated higher than non-edges: Mann-Whitney U = 607784, one-sided p = 1.76e-60
Share of annotated edges rated strong (>= 6): 49.0%


## 4. How reliable are the expert annotations?

Krippendorff's alpha (interval) over the raw ratings: 1 means perfect agreement, 0 means no better than chance. It works with a different number of raters per pair and does not need to know which rater gave which score. The same measure family as the Cohen's kappa used for our own graph in Objective 1.

In [7]:
def krippendorff_alpha_interval(units):
    # units: list of rating lists (one list per annotated pair); units with < 2 ratings are skipped.
    units = [np.asarray(u, dtype=float) for u in units if len(u) >= 2]
    values = np.concatenate(units)
    n = len(values)
    observed = sum(2 * len(u) * ((u - u.mean()) ** 2).sum() / (len(u) - 1) for u in units) / n
    expected = 2 * ((values - values.mean()) ** 2).sum() / (n - 1)
    return 1 - observed / expected

def parse(raw):
    return [float(x) for x in str(raw).split("_") if x.strip() != ""]

reliability = pd.Series({
    col: round(krippendorff_alpha_interval(rel[f"{col}_raw"].map(parse)), 3)
    for col in ("Prerequisite", "Similarity", "Difficulty")
}, name="Krippendorff alpha (interval)")
raters = rel["Prerequisite_raw"].map(lambda r: len(parse(r)))
print(f"raters per pair: min {raters.min()}, median {int(raters.median())}, max {raters.max()}")
reliability

raters per pair: min 3, median 7, max 15


,Krippendorff alpha (interval)
Prerequisite,0.468
Similarity,0.628
Difficulty,0.443


## 5. Export for GAT training

In [8]:
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())
missing = sorted(set(G) - set(exercise_index))
assert not missing, f"exercises missing from notebook 01's index: {missing[:5]}"

edge_index = np.array(
    [[exercise_index[p] for p, _ in G.edges()], [exercise_index[c] for _, c in G.edges()]],
    dtype=np.int64,
)  # row 0 = prerequisite, row 1 = dependent exercise (message flows prerequisite -> dependent)

# Similarity graph for the ablation "prerequisite edges vs similarity edges" (notebook 06):
# annotated pairs with high similarity, in both directions, in the same numbering.
sim = rel[(rel["Similarity_avg"] >= 6) & rel.Exercise_A.isin(exercise_index) & rel.Exercise_B.isin(exercise_index)]
sim_pairs = sorted({(exercise_index[a], exercise_index[b]) for a, b in zip(sim.Exercise_A, sim.Exercise_B)}
                   | {(exercise_index[b], exercise_index[a]) for a, b in zip(sim.Exercise_A, sim.Exercise_B)})
similarity_edge_index = np.array(sim_pairs, dtype=np.int64).T

np.save(OUT_DIR / "edge_index.npy", edge_index)
np.save(OUT_DIR / "similarity_edge_index.npy", similarity_edge_index)
pd.DataFrame(list(G.edges()), columns=["prerequisite", "exercise"]).to_csv(OUT_DIR / "edges.csv", index=False)
print("prerequisite edges:", edge_index.shape[1], "  similarity edges:", similarity_edge_index.shape[1])

prerequisite edges: 933   similarity edges: 1219


In [9]:
import hashlib, platform
from datetime import datetime, timezone

stats_out = {
    "graph": {k: (int(v) if isinstance(v, (np.integer, int)) else v) for k, v in graph_stats.items()},
    "direction_check": summary.reset_index().to_dict(orient="records"),
    "edges_vs_non_edges_mann_whitney_p": float(p),
    "annotation_reliability_alpha": reliability.to_dict(),
}
(OUT_DIR / "graph_stats.json").write_text(json.dumps(stats_out, indent=2, default=str))

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": {"SEED": SEED, "similarity_threshold": 6},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "networkx": nx.__version__},
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "created_at": "2026-10-07T12:10:57.469227+00:00",
  "config": {
    "SEED": 42,
    "similarity_threshold": 6
  },
  "versions": {
    "python": "3.13.15",
    "pandas": "2.2.3",
    "numpy": "2.1.3",
    "networkx": "3.7"
  },
  "sha256": {
    "edge_index.npy": "1e50374981b2ab3b3028df577d95073f064431ba32634a2e7c4f5f652b7321d7",
    "edges.csv": "50522c9613b748f54b126d7946be881eb05b1a99158f0b3559ae0564a8c0b725",
    "graph_stats.json": "88078d93a19c2de20490747bc3e9089aca9559f8e47473788ec7b47a142176f8",
    "similarity_edge_index.npy": "3107508e94953578f12d00921135278131c1e2535ab45c843ca1e0301eb711a5"
  }
}


## 6. (Optional) Junyi graph in Neo4j, for visuals only

Imported with its **own labels** (`:JunyiExercise`, `:JUNYI_PREREQUISITE_OF`), so it never mixes with the C1 concept graph (`:Concept`) if both live in the same Aura database. The password is typed in, never saved in the notebook.

In [10]:
IMPORT_TO_NEO4J = False

if IMPORT_TO_NEO4J:
    !{sys.executable} -m pip install -q neo4j
    from getpass import getpass
    from neo4j import GraphDatabase
    uri = input("Neo4j URI (neo4j+s://...): ")
    driver = GraphDatabase.driver(uri, auth=("neo4j", getpass("Neo4j password: ")))
    nodes = [{"id": n, "area": area.get(n)} for n in G]
    rels = [{"p": p, "c": c} for p, c in G.edges()]
    with driver.session() as s:
        s.run("CREATE CONSTRAINT junyi_id IF NOT EXISTS FOR (e:JunyiExercise) REQUIRE e.id IS UNIQUE")
        s.run("UNWIND $rows AS r MERGE (e:JunyiExercise {id: r.id}) SET e.area = r.area", rows=nodes)
        s.run("UNWIND $rows AS r MATCH (p:JunyiExercise {id: r.p}), (c:JunyiExercise {id: r.c}) "
              "MERGE (p)-[:JUNYI_PREREQUISITE_OF]->(c)", rows=rels)
    driver.close()
    print("Imported. In Neo4j Browser: MATCH p=()-[:JUNYI_PREREQUISITE_OF]->() RETURN p LIMIT 200")

In [11]:
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive("/content/graph_outputs", "zip", OUT_DIR))
else:
    print("Outputs stored in", OUT_DIR)

Outputs stored in /content/drive/MyDrive/adaptlearn-c1/junyi_processed/graph


## For the report (Appendix B, corrected)

Report from this notebook: live exercises, edges after splitting, removed edges by reason, no-cycle result, the direction table, the Mann-Whitney result and the alpha values. These replace the earlier counts (742 cells, 357 strong pairs) that did not join the two files.

Before committing: **Clear all outputs**.